# Phase 3a: fine-tune whisper-medium with LoRA

Same recipe as whisper-small (notebook 03): LoRA, the segmented pieces from notebook 02, dev WER every 100 steps,
early stopping, best checkpoint by dev WER. Only the model and the batch layout differ (see the config).

Run on a Colab L4 (Runtime > Change runtime type > L4 GPU). Before starting, run
`bash scripts/make_colab_bundle.sh` locally and upload the new `cyasr_bundle.zip` to `MyDrive/cyasr/`,
replacing the old one. `MyDrive/cyasr/segments.zip` must exist (made at the end of the whisper-small run).

The real run writes straight to `MyDrive/cyasr/runs/` with `resume: true`: after a disconnect, reconnect and run
the cells again (skip the smoke test), and training continues from the last checkpoint. `timing.json` only
covers the last session after a resume, so note the elapsed time of earlier sessions by hand.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CONFIG = 'configs/finetune_whisper-medium.yaml'
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_SEGMENTS_ZIP = '/content/drive/MyDrive/cyasr/segments.zip'
DRIVE_RUNS = '/content/drive/MyDrive/cyasr/runs'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!ls cyasr/train.py {CONFIG}  # fails if the bundle is the old one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch is already installed on Colab; this adds peft and the other requirements
!pip install -q -r requirements.txt
# Colab ships an old torchao that peft rejects ("only versions above 0.16.0"); we don't use it
!pip uninstall -y -q torchao

In [ ]:
# The segmented pieces as one zip: one large file copies from Drive in about a minute,
# 1,227 small files took over 15 minutes.
!cp {DRIVE_SEGMENTS_ZIP} /content/segments.zip
!mkdir -p data && unzip -q -o /content/segments.zip -d data/
import pandas as pd
seg = pd.read_csv('data/segments/segments.tsv', sep='\t')
print(seg.groupby('split').agg(pieces=('id', 'count'), hours=('duration_s', lambda s: round(s.sum() / 3600, 2))))
!ls data/segments/train | wc -l  # expect 820
!ls data/segments/dev | wc -l    # expect 407

In [ ]:
# Smoke test: same settings, but 4 steps and 8 dev pieces, saved locally (not on Drive).
# Look at peak_gpu_memory_gb in the timing line: it shows whether the batch fits on this GPU.
import yaml
!rm -rf runs/finetune_smoke
cfg = yaml.safe_load(open(CONFIG))
cfg.update(run_name='smoke', max_steps=4, eval_steps=2, warmup_steps=1, max_eval_samples=8)
yaml.safe_dump(cfg, open('/content/smoke.yaml', 'w'), allow_unicode=True)
!python -m cyasr.train --config /content/smoke.yaml

In [ ]:
# The real run, written straight to Drive with resume on.
# After a disconnect: reconnect, run the cells above again (skip the smoke test), then this one.
import yaml
cfg = yaml.safe_load(open(CONFIG))
cfg.update(output_dir=DRIVE_RUNS, resume=True)
yaml.safe_dump(cfg, open('/content/run.yaml', 'w'), allow_unicode=True)
!mkdir -p {DRIVE_RUNS}
print('run folder:', f"{DRIVE_RUNS}/finetune_{cfg['run_name']}")
!python -m cyasr.train --config /content/run.yaml

In [ ]:
# Dev WER over training, and the time it took
import json, yaml
RUN = f"{DRIVE_RUNS}/finetune_" + yaml.safe_load(open(CONFIG))['run_name']
log = json.load(open(f'{RUN}/train_log.json'))
train_loss = {e['step']: e['loss'] for e in log if 'loss' in e}
print('step  train_loss  dev_loss  WER_basic  WER_nodiac  CER_nodiac')
for e in log:
    if 'eval_wer_basic' in e:
        tl = train_loss.get(e['step'], float('nan'))
        print(f"{e['step']:4}  {tl:10.3f}  {e['eval_loss']:8.3f}  {e['eval_wer_basic']:9.1%}  "
              f"{e['eval_wer_nodiac']:10.1%}  {e['eval_cer_nodiac']:10.1%}")
print(json.dumps(json.load(open(f'{RUN}/timing.json')), indent=2))

In [ ]:
# Everything is already on Drive. Check what is there and how big it is.
!ls {RUN}
!du -sh {RUN}/*
# Once best/ has been evaluated and you're happy with it, free the space taken by the checkpoints:
# !rm -rf {RUN}/checkpoints